<div dir="rtl">

# کارگاه جلسه ۲۳ — روش‌های گروهی

هر بخش مستقل است. نوشتن این کدها در امتحان پرسیده نمی‌شود.

1. انباشتن مدل‌ها (Stacking) روی تایتانیک
2. تمرین عملی کلاس: درخت، جنگل تصادفی و Gradient Boosting برای مصرف سوخت

</div>

<div dir="rtl">

## ۱. انباشتن مدل‌ها (Stacking)

در **انباشتن**، یک مدل دوم (اینجا رگرسیون لجستیک) یاد می‌گیرد پیش‌بینی‌های مدل‌های پایه را چگونه ترکیب کند. پیش‌بینی‌های مدل‌های پایه برای آموزش مدل دوم با اعتبارسنجی متقابل داخلی ساخته می‌شوند تا نشت رخ ندهد.

</div>

In [1]:
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier

titanic = sns.load_dataset("titanic")
num = ["age", "fare", "sibsp", "parch", "pclass"]
cat = ["sex", "embarked"]
X_train, X_test, y_train, y_test = train_test_split(titanic[num + cat], titanic["survived"],
                                                    test_size=0.2, stratify=titanic["survived"],
                                                    random_state=42)
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat)])
base = [("logistic", LogisticRegression()), ("knn", KNeighborsClassifier(17)),
        ("forest", RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=42)),
        ("hgb", HistGradientBoostingClassifier(random_state=42))]
stack = StackingClassifier(base, final_estimator=LogisticRegression(), cv=5)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in base + [("stacking", stack)]:
    print(f"{name:9s} CV accuracy={cross_val_score(make_pipeline(preprocess, clf), X_train, y_train, cv=cv).mean():.3f}")
model = make_pipeline(preprocess, stack).fit(X_train, y_train)
weights = model[-1].final_estimator_.coef_[0]
print("meta-model weights:", {n: round(float(w), 2) for (n, _), w in zip(base, weights)})

logistic  CV accuracy=0.796
knn       CV accuracy=0.826


forest    CV accuracy=0.822


hgb       CV accuracy=0.813


stacking  CV accuracy=0.813


meta-model weights: {'logistic': 1.09, 'knn': 1.3, 'forest': 1.66, 'hgb': 1.45}


<div dir="rtl">

## ۲. تمرین عملی کلاس (۲۵ دقیقه): مصرف سوخت

داده mpg را با شش ویژگی عددی جلسه ۱۹ (cylinders، displacement، horsepower، weight، acceleration، model_year) و همان تقسیم (۲۰ درصد آزمون، `random_state=42`) و `KFold(5, shuffle=True, random_state=42)` به کار ببرید.

1. RMSE اعتبارسنجی متقابل درخت رگرسیون (حداقل ۱۰ نمونه در برگ)، جنگل تصادفی (۵۰۰ درخت) و Gradient Boosting را مقایسه کنید.
2. برای جنگل تصادفی `max_features` را از میان ۱، ۲، ۳ و ۶ (یعنی همه ویژگی‌ها، همان بگینگ) انتخاب کنید. کدام بهتر است و چرا؟
3. بهترین مدل را روی آزمون بسنجید و با Ridge درجه ۳ جلسه ۱۹ (RMSE آزمون ۲.۵۸) مقایسه کنید.
4. اهمیت جایگشتی ویژگی‌های بهترین مدل را روی آزمون حساب کنید.

</div>

In [2]:
# Write your solution here

<div dir="rtl">

### پاسخ تمرین

</div>

In [3]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import root_mean_squared_error

mpg = sns.load_dataset("mpg").dropna(subset=["horsepower"])
features = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
X_train, X_test, y_train, y_test = train_test_split(mpg[features], mpg["mpg"], test_size=0.2, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {"Tree (min_leaf=10)": DecisionTreeRegressor(min_samples_leaf=10, random_state=42),
          "Random forest": RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1),
          "Gradient boosting": GradientBoostingRegressor(random_state=42)}
for name, m in models.items():
    rmse = -cross_val_score(m, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error").mean()
    print(f"{name:20s} CV RMSE={rmse:.3f}")

gs = GridSearchCV(RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1),
                  {"max_features": [1, 2, 3, 6]}, cv=kf, scoring="neg_root_mean_squared_error").fit(X_train, y_train)
for mf, s in zip([1, 2, 3, 6], gs.cv_results_["mean_test_score"]):
    print(f"max_features={mf}  CV RMSE={-s:.3f}")
best = gs.best_estimator_
print(f"best={gs.best_params_}  test RMSE={root_mean_squared_error(y_test, best.predict(X_test)):.3f}")
perm = permutation_importance(best, X_test, y_test, n_repeats=20, random_state=42,
                              scoring="neg_root_mean_squared_error")
print(pd.Series(perm.importances_mean, index=features).sort_values(ascending=False).round(3))

Tree (min_leaf=10)   CV RMSE=3.283


Random forest        CV RMSE=2.868
Gradient boosting    CV RMSE=2.892


max_features=1  CV RMSE=2.925
max_features=2  CV RMSE=2.907
max_features=3  CV RMSE=2.884
max_features=6  CV RMSE=2.868
best={'max_features': 6}  test RMSE=2.388


model_year      2.199
displacement    1.260
weight          0.990
horsepower      0.710
acceleration    0.193
cylinders       0.041
dtype: float64


<div dir="rtl">

### پاسخ‌های کلیدی

- انباشتن (بخش ۱) با دقت اعتبارسنجی ۰.۸۱۳ از بهترین عضو (17-NN با ۰.۸۲۶) بهتر نشد؛ مدل دوم به جنگل تصادفی بیشترین وزن را داد. روی داده کوچک و با اعضای همبسته، انباشتن سود چندانی ندارد.
- RMSE اعتبارسنجی: درخت **۳.۲۸**، جنگل تصادفی **۲.۸۷** و Gradient Boosting **۲.۸۹**. هر دو روش گروهی بسیار بهتر از درخت تکی‌اند.
- تنظیم `max_features`: ۱ (۲.۹۳)، ۲ (۲.۹۱)، ۳ (۲.۸۸) و ۶ (۲.۸۷). اینجا بگینگ ساده (همه ویژگی‌ها) کمی بهتر است؛ با فقط شش ویژگی که چند تای آن‌ها (وزن، حجم و قدرت موتور) تقریباً هم‌ارزند، درخت‌ها حتی بدون تصادفی‌سازی ویژگی متنوع‌اند. تفاوت‌ها کوچک‌اند.
- RMSE آزمون جنگل تصادفی **۲.۳۹** است؛ بهتر از Ridge درجه ۳ جلسه ۱۹ (۲.۵۸)، بدون ساخت دستی ویژگی‌های چندجمله‌ای.
- اهمیت جایگشتی روی آزمون: سال ساخت (۲.۲۰)، حجم موتور (۱.۲۶)، وزن (۰.۹۹) و قدرت موتور (۰.۷۱)؛ تعداد سیلندر تقریباً بی‌اثر است، چون اطلاعاتش در حجم موتور هست.

</div>